# 🤝 Réseau de solidarité locale IA
Notebook Colab prêt à utiliser avec données imaginées.


In [ ]:
!pip install pandas numpy scikit-learn plotly -q


In [ ]:
import pandas as pd
import numpy as np
import math
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


In [ ]:
besoins = pd.DataFrame([
    ['B001','Famille Diallo','Villeray',45.5501,-73.6287,'nourriture','Besoin d’un panier alimentaire avec produits frais et couches',5,4],
    ['B002','Mme Tremblay','Plateau',45.5215,-73.5804,'entraide','Besoin d’aide pour faire l’épicerie et monter les sacs',3,1],
    ['B003','Collectif Jeunes Hochelaga','Hochelaga',45.5468,-73.5455,'logement','Recherche hébergement temporaire pour deux jeunes',5,2],
], columns=['id_besoin','nom','quartier','latitude','longitude','categorie','description','urgence','beneficiaires'])

ressources = pd.DataFrame([
    ['R001','Banque alimentaire Villeray','Villeray',45.5489,-73.6254,'nourriture','Paniers alimentaires, couches, produits frais',25,'oui'],
    ['R002','Voisins Solidaires Plateau','Plateau',45.5230,-73.5820,'entraide','Bénévoles pour courses, visites et accompagnement',10,'oui'],
    ['R003','Hébergement Horizon','Hochelaga',45.5481,-73.5432,'logement','Lits temporaires pour jeunes adultes',4,'oui'],
    ['R004','Cuisine Collective CDN','Côte-des-Neiges',45.4970,-73.6180,'nourriture','Repas préparés et paniers d’urgence',18,'oui'],
], columns=['id_ressource','organisation','quartier','latitude','longitude','categorie','description','capacite','disponibilite'])

besoins


In [ ]:
def haversine_km(lat1, lon1, lat2, lon2):
    r = 6371
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)
    a = math.sin(dphi/2)**2 + math.cos(phi1)*math.cos(phi2)*math.sin(dlambda/2)**2
    return 2*r*math.atan2(math.sqrt(a), math.sqrt(1-a))

def calculer_matches(besoins, ressources, top_n=3):
    besoins = besoins.copy(); ressources = ressources.copy()
    besoins['texte'] = besoins['categorie'] + ' ' + besoins['description']
    ressources['texte'] = ressources['categorie'] + ' ' + ressources['description']
    vectorizer = TfidfVectorizer()
    corpus = pd.concat([besoins['texte'], ressources['texte']], ignore_index=True)
    vecteurs = vectorizer.fit_transform(corpus)
    sim = cosine_similarity(vecteurs[:len(besoins)], vecteurs[len(besoins):])
    lignes = []
    for i, besoin in besoins.iterrows():
        candidats = []
        for j, ressource in ressources.iterrows():
            distance = haversine_km(besoin.latitude, besoin.longitude, ressource.latitude, ressource.longitude)
            score = (
                0.35 * sim[i, j] +
                0.25 * (besoin.categorie == ressource.categorie) +
                0.20 * max(0, 1 - distance / 15) +
                0.10 * (besoin.urgence / 5) +
                0.10 * min(1, ressource.capacite / max(1, besoin.beneficiaires))
            )
            candidats.append([besoin.id_besoin, besoin.nom, ressource.organisation, ressource.categorie, round(distance,2), round(score*100,1)])
        lignes += sorted(candidats, key=lambda x: x[-1], reverse=True)[:top_n]
    return pd.DataFrame(lignes, columns=['id_besoin','nom','organisation','categorie_ressource','distance_km','score'])


In [ ]:
matches = calculer_matches(besoins, ressources)
matches


## Prochaine étape
Remplacez les données imaginées par des données réelles d'organismes, après consentement et validation humaine.
